# Part 5: Parameter estimation with PEtab.jl

This notebook introduces how to combine Catalyst with [PEtab.jl](https://github.com/sebapersson/PEtab.jl) to define and solve parameter estimation problems of the form

$$
\min_{\mathbf{x} \in \mathbb{R}^n} \ell(\mathbf{x})
\quad \text{s.t.} \quad
\mathbf{lb} \leq \mathbf{x} \leq \mathbf{ub},
$$

where $\ell$ is the likelihood function, $\mathbf{x}$ are the unknown model parameters, and $\mathbf{lb}$ and $\mathbf{ub}$ are the lower and upper parameter bounds.

In brief, PEtab.jl provides a structured way to define all components of a parameter estimation problem and offers functionality for model fitting. In this notebook, we illustrate the workflow using simulated data from a Michaelis-Menten model.

## Part 5.1: Create a parameter estimation problem

A PEtab parameter estimation problem consists of five parts:

1. **Dynamic model**: a Catalyst `ReactionSystem`.
2. **Observables**: `PEtabObservable`s that map model states and parameters to measurements.
3. **Parameters**: `PEtabParameter`s that specify which parameters are estimated.
4. **Simulation conditions** *(optional)*: `PEtabCondition`s that define experimental conditions for the measurements.
5. **Measurements**: measurement data stored as a `DataFrame` in the required PEtab format.

### 5.1.1: Define the dynamic model

The dynamic model is provided as a `ReactionSystem`. Default values can (and should) be defined directly in the system:

In [ ]:
using Catalyst
rn = @reaction_network begin
    @parameters S0 c3=3.0
    @species begin
        S(t) = S0
        E(t) = 25.0
        SE(t) = 0.0
        P(t) = 0.0
    end
    @observables begin
        obs1 ~ P
        obs2 ~ S + E
    end
    c1, S + E --> SE
    c2, SE --> S + E
    c3, SE --> P + E
end

Constant parameters, such as `c3`, and parameters that appear in initial conditions, such as `S0`, must be declared in `@parameters`. Parameters to be estimated (`c1`, `c2`, and `S0` in this example) do not need assigned values in the `ReactionSystem`. In addition, any observable used to link model output to measurement data can be defined in the system and then referenced when creating the corresponding `PEtabObservable`.

### 5.1.2 Define observables

To map model outputs to measurements, each measured quantity must define an **observable formula**, and a **noise formula**. These are specified with `PEtabObservable`. If observables are already defined in the model system, such as `obs1` and `obs2` above, they can be referenced directly by name.

For example, suppose `obs1 = P` is measured with known noise `sigma = 1.0`:

In [ ]:
using PEtab
petab_obs1 = PEtabObservable(:petab_obs1, :obs1, 1.0)

The first argument is the `observable_id`, which is referenced later in the measurement table. The second argument, `:obs1`, refers to the observable defined in the model system, and the third argument specifies the noise parameter. By default, measurements are assumed to be Normally distributed, as shown in the printed summary.

The observable formula can also be defined directly in `PEtabObservable`, and the noise parameter can be estimated as part of the parameter estimation problem. For example, suppose `obs2 = S + E` is measured with an unknown noise parameter `sigma`:

In [ ]:
t = Catalyst.default_t()
@parameters sigma
@variables S(t) E(t)
petab_obs2 = PEtabObservable(:petab_obs2, S + E, sigma)

Noise and observable formulas can be any Julia expression. Finally, all observables should be collected into a `Vector`:

In [ ]:
model_obs = [petab_obs1, petab_obs2]

### Part 5.1.3: Define parameters to estimate

Parameters to be estimated are specified with `PEtabParameter`. Every estimated parameter must be defined this way. For example, to estimate `c1`:

In [ ]:
p_c1 = PEtabParameter(:c1)

By default, parameters are assigned bounds `[1e-3, 1e3]` and estimated on a `log10` scale (can be changed via keyword arguments), as this typically improves performance. Parameters can also be assigned priors. For example, to assign a `LogNormal(2.5, 0.3)` prior to `c2`:

In [ ]:
using Distributions
p_c2 = PEtabParameter(:c2; prior = LogNormal(2.5, 0.3))

All parameters to estimate should be defined similarly, and collected into a `Vector`:

In [ ]:
p_sigma = PEtabParameter(:sigma)
pest = [p_c1, p_c2, p_sigma]

### Part 5.1.4: Simulation conditions

Measurements are often collected under different experimental conditions. In the model, this corresponds to simulating with different initial values and/or parameter values. Such condition-specific settings are defined with `PEtabCondition`.

For example, suppose we have two experiments, `cond1` and `cond2`, with different initial concentrations of the substrate `S`:

In [ ]:
conditions = [
    PEtabCondition(:cond1, :S0 => 100.0)
    PEtabCondition(:cond2, :S0 => 50.0)
]

The first argument is a unique condition identifier. The remaining arguments specify condition-specific assignments. Here, we assign a value to `S0`, which was defined above to set the initial value of species `S`.

### Part 5.1.5: Measurement data

Measurement data is provided as a tidy `DataFrame`, where each row corresponds to a single observation. Thus, each row must link one measurement to an observable and a simulation condition. Repeated measurements at the same time point are represented by separate rows. Overall, the data should have the following form:

| simulation_id (str) | obs_id (str) | time (float) | measurement (float) |
| ------------------- | ------------ | ------------ | ------------------- |
| ...                 | ...          | ...          | ...                 |

For this example, we simulate measurement data for both simulation conditions:

In [ ]:
using OrdinaryDiffEqRosenbrock, DataFrames
# Simulate with 'true' parameters
u0 = [:E => 25.0, :SE => 0.0, :P => 0.0]
tspan = (0.0, 10.0)

ps_cond1 = [:c1 => 1.0, :c2 => 10.0, :c3 => 3.0, :S0 => 100.0]
ode_cond1 = ODEProblem(rn, u0, tspan, ps_cond1)
sol_cond1 = solve(
    ode_cond1, Rodas5P(); abstol = 1e-8, reltol = 1e-8, saveat = 0:0.5:10.0
)
obs1_cond1 = sol_cond1[:P] .+ randn(length(sol_cond1))
obs2_cond1 = sol_cond1[:E] .+ sol_cond1[:S] .+ randn(length(sol_cond1)) .* 0.3
df1 = DataFrame(
    simulation_id = "cond1", obs_id = "petab_obs1", 
    time = sol_cond1.t, measurement = obs1_cond1
)
df2 = DataFrame(
    simulation_id = "cond1", obs_id = "petab_obs2", 
    time = sol_cond1.t, measurement = obs2_cond1
)

ps_cond2 = [:c1 => 1.0, :c2 => 10.0, :c3 => 3.0, :S0 => 50.0]
ode_cond2 = ODEProblem(rn, u0, tspan, ps_cond2)
sol_cond2 = solve(
    ode_cond2, Rodas5P(); abstol = 1e-8, reltol = 1e-8, saveat = 0:0.5:10.0
)
obs1_cond2 = sol_cond2[:P] .+ randn(length(sol_cond2))
obs2_cond2 = sol_cond2[:E] .+ sol_cond2[:S] .+ randn(length(sol_cond2)) .* 0.3
df3 = DataFrame(
    simulation_id = "cond2", obs_id = "petab_obs1", 
    time = sol_cond2.t, measurement = obs1_cond2
)
df4 = DataFrame(
    simulation_id = "cond2", obs_id = "petab_obs2", 
    time = sol_cond2.t, measurement = obs2_cond2
)

measurements = vcat(df1, df2, df3, df4)

### Part 5.1.6: Build the parameter estimation problem

Given a model, observables, estimated parameters, simulation conditions, and measurement data, the parameter estimation problem is constructed in two steps: first create a `PEtabModel`, then create a `PEtabODEProblem`.

In [ ]:
model_rn = PEtabModel(
    rn, model_obs, measurements, pest; simulation_conditions = conditions
)
petab_prob = PEtabODEProblem(model_rn)


Calling `describe` prints a summary of the problem:

In [ ]:
describe(petab_prob)

The default options are usually well-performing, and tuned based on an extensive benchmarks. That said, they can still be changed. For example, the ODE-solver can be provided as:


In [ ]:
petab_prob = PEtabODEProblem(
    model_rn; odesolver = ODESolver(Rodas5P())
)

A `PEtabODEProblem` contains everything needed for parameter estimation, so as a next step lets fit the model to data!

## Part 5.2: Parameter estimation

PEtab.jl provides high-level wrappers for parameter estimation with several optimizers, including Optim.jl, Ipopt.jl, [Fides.jl](https://github.com/fides-dev/Fides.jl), and Optimization.jl. In this section, we use Fides.jl to estimate parameters from an initial guess `x0`, and then apply multistart optimization to reduce the risk of converging to a local minimum.

### Part 5.2.1: Single-start parameter estimation

Numerical optimizers require an initial guess `x0` in the parameter order defined by the `PEtabODEProblem`. To reduce bias, it is often useful to sample `x0` randomly within the parameter bounds. This can be done with `get_startguesses`:

In [ ]:
using StableRNGs
rng = StableRNG(42) # for reproducibility
x0 = get_startguesses(rng, petab_prob, 1)

`x0` is a `ComponentArray`, so parameters can be accessed by name. Parameters estimated on a log scale use names with a corresponding prefix, such as `log10_c1`, and values must be set on that scale. For example, to set `c1 = 10.0`:


In [ ]:
x0.log10_c1 = log10(10.0)
x0[:log10_c1] = log10(10.0) # alternatively 

Given `x0`, we can estimate the parameters. For this small example with three parameters, we use the Newton trust-region method from [Fides.jl](https://github.com/fides-dev/Fides.jl), with the Hessian provided by `petab_prob` via `CustomHessian`:

In [ ]:
using Fides
res = calibrate(petab_prob, x0, Fides.CustomHessian())

The printed summary includes optimization statistics such as the final objective value `fmin`, which in PEtab.jl corresponds to the negative log-likelihood. The estimated parameter values are available as:

In [ ]:
res.xmin

Finally, it is often useful to assess the fit by plotting the model predictions against the measurement data:

In [ ]:
using Plots
p1 = plot(res, petab_prob; condition = :cond1)
p2 = plot(res, petab_prob; condition = :cond2)
plot(p1, p2, layout = (1, 2), size = (1200, 400))

In this case, the fit is excellent. Even so, it is good practice to use a global strategy such as multistart parameter estimation to reduce the risk of converging to a local minimum.

### Part 5.2.1: Multi-start parameter estimation

In multistart estimation, the optimizer is run from multiple starting points to reduce the risk of converging to a local minimum. PEtab.jl provides `calibrate_multistart`. For example, to run `n = 50` starts:

In [ ]:
using Fides, StableRNGs
rng = StableRNG(42)
ms_res = calibrate_multistart(rng, petab_prob, Fides.CustomHessian(), 50)

A useful diagnostic is the waterfall plot, which shows the final objective value for each start (sorted best to worst):

In [ ]:
plot(ms_res; plot_type = :waterfall)

The plateaus correspond to different optima. In this case, the objective values are nearly identical across runs, suggesting that all starts converged to the same optimum. Finally, the simulation for the best-fit parameters can be plotted against the data as:

In [ ]:
p1 = plot(res, petab_prob; condition = :cond1)
p2 = plot(res, petab_prob; condition = :cond2)
plot(p1, p2, layout = (1, 2), size = (1200, 400))

### Part 5.4: Do it yourself: Condition-specific parameters

PEtab.jl supports several features beyond those covered above, including condition-specific parameters. Specifically, above the initial value of species `S`, `S0`, was treated as known. In practice, parameters that vary across conditions may also be unknown and need to be estimated.

Using [this tutorial](https://sebapersson.github.io/PEtab.jl/stable/tutorials/define_problem/condition_parameters), modify the example above so that `S0` is estimated separately for each condition.

In [ ]:
# Insert code here!

## Part 5.5: Bonus: The PEtab standard format

PEtab.jl is built around the [PEtab standard](https://petab.readthedocs.io/en/latest/) for parameter estimation problems. Models in this format can be imported directly into PEtab.jl:

In [ ]:
using OrdinaryDiffEqBDF
path_yaml = joinpath(
    @__DIR__, "assets", "Boehm_JProteomeRes2014", "Boehm_JProteomeRes2014.yaml"
)
model_boehm = PEtabModel(path_yaml)
petab_prob_boehm = PEtabODEProblem(
    model_boehm; odesolver = ODESolver(QNDF())
)

The resulting `PEtabODEProblem` can then be used for parameter estimation as shown above.

### Part 5.5.1: Do it yourself: Parameter estimation

For the Boehm model, a good optimizer choice is `Fides.BFGS`. Run a 100-start multistart parameter estimation and evaluate the resulting model fit.

## Where to go next

This notebook covers a subset of the functionality available in PEtab.jl. For additional features for defining parameter estimation problems, see the following tutorials:

- [Simulation conditions](https://sebapersson.github.io/PEtab.jl/stable/tutorials/define_problem/simulation_conditions): Measurements collected under different experimental conditions, for example with different initial values.
- [Pre-equilibration](https://sebapersson.github.io/PEtab.jl/stable/tutorials/define_problem/pre_equilibration): Enforce a steady state before comparing the model to data.
- [Simulation condition-specific parameters](https://sebapersson.github.io/PEtab.jl/stable/tutorials/define_problem/condition_parameters): Estimate parameters that take different values across simulation conditions.
- [Observable and noise parameters](https://sebapersson.github.io/PEtab.jl/stable/tutorials/define_problem/observable_noise_parameters): Define observable and noise parameters in `PEtabObservable` formulas that are not part of the model system, such as scaling or offset terms, optionally in a time-point-specific way.
- [Events/callbacks](https://sebapersson.github.io/PEtab.jl/stable/tutorials/define_problem/events): Handle time- or state-triggered events and callbacks.
- [Supported model systems](https://sebapersson.github.io/PEtab.jl/stable/tutorials/define_problem/model_system): Overview of supported ways to define the dynamic model, including `ReactionSystem`, `ODESystem`, and `ODEProblem`.

This notebook also introduced parameter estimation with Fides.jl. For a more detailed overview of the estimation functionality in PEtab.jl, see:

- [Parameter estimation extended tutorial](https://sebapersson.github.io/PEtab.jl/stable/tutorials/parameter_estimation/extended_tutorial): Extended coverage of estimation workflows, including multistart optimization and Optimization.jl integration.